In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import h5py
from scipy.fft import fftn, ifftn, fftfreq
import illustris_python as il
from matplotlib.colors import LogNorm

import matplotlib.animation as animation
import matplotlib.patches as mpatches

In [ ]:
with h5py.File('nexus_outputs.h5', 'r') as f:
    norm_dens = f['z_0.0']['density'][:]
    voids = f['z_0.0']['voids'][:]
    filaments = f['z_0.0']['filaments'][:]
    nodes = f['z_0.0']['nodes'][:]
    walls = f['z_0.0']['walls'][:]
    print(f"Suessfully Loaded Stuff")

In [ ]:
norm_dens = np.transpose(norm_dens, (1, 2, 0))
voids = np.transpose(voids, (1, 2, 0))
filaments = np.transpose(filaments, (1, 2, 0))
nodes = np.transpose(nodes, (1, 2, 0))
walls = np.transpose(walls, (1, 2, 0))

In [ ]:
basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"

coords_0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['Coordinates'])/1000 #in cMpc/h

In [ ]:
L = 35.0 # cMpc/h
res = 128

dl  = L / res
print (dl)

In [ ]:
slice_index = 1

z_min = slice_index * dl
z_max = (slice_index + 1) * dl
mask = (coords_0[:,2] > z_min) & (coords_0[:,2] < z_max)

x = coords_0[:, 0][mask]
y = coords_0[:, 1][mask]
z = coords_0[:, 2][mask]

fig = plt.figure(figsize=(12,4))

ax1 = fig.add_subplot(1,2,1)
im1 = ax1.imshow(np.log10(norm_dens[:,:,slice_index]),origin = "lower",cmap="inferno", extent = [0,35,0,35])
ax1.set_xlabel("cMpc/h")
ax1.set_ylabel("cMpc/h")
ax1.set_aspect('equal')
ax1.contour(filaments[:,:,slice_index], extent = [0,35,0,35],alpha=0.2)
fig.colorbar(im1, ax = ax1, label = r"$log_{10}(1+\delta)$")

ax2 = fig.add_subplot(1,2,2)
counts, xedges, yedges, im2 = ax2.hist2d(x, y, bins = 500, cmap = "inferno", norm = LogNorm())
ax2.set_xlabel("cMpc/h")
ax2.set_ylabel("cMpc/h")
ax2.set_aspect('equal')
ax2.contour(filaments[:,:,slice_index], extent = [0,35,0,35],alpha=0.2)
fig.colorbar(im2, ax = ax2, label = "Counts")

plt.tight_layout();

In [ ]:
import matplotlib.animation as animation
import matplotlib.patches as mpatches

d_vmin = np.min(np.log10(norm_dens))
d_vmax = np.max(np.log10(norm_dens))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

dummy_im = ax1.imshow(np.zeros((2, 2)), cmap="Grays_r", vmin=d_vmin, vmax=d_vmax)
fig.colorbar(dummy_im, ax=ax1, label=r"$log_{10}(1+\delta)$")
counts, xedges, yedges, dummy_hist = ax2.hist2d([0, 1], [0, 1], bins=2, cmap="Grays_r", norm=LogNorm(vmin=1, vmax=10**4))
fig.colorbar(dummy_hist, ax=ax2, label="Counts")

plt.tight_layout()

filament_patch = mpatches.Patch(color='green', alpha=0.3, label='Filaments')
wall_patch = mpatches.Patch(color='blue', alpha=0.3, label='Walls')
node_patch = mpatches.Patch(color='red', alpha=0.3, label='Nodes')
fig.legend(handles=[filament_patch,wall_patch, node_patch],loc='upper center', ncol=1, frameon=True)
fig.subplots_adjust(top=0.85)

def update(frame):

    ax1.clear()
    ax2.clear()

    slice_index = frame

    im1 = ax1.imshow(np.log10(norm_dens[:,:,slice_index]),origin = "lower",cmap="Grays_r", extent = [0,35,0,35], vmin = d_vmin, vmax = d_vmax)
    ax1.set_xlabel("cMpc/h")
    ax1.set_ylabel("cMpc/h")
    ax1.set_aspect('equal')
    ax1.contour(filaments[:,:,slice_index], extent = [0,35,0,35],alpha=0.1, colors='green')
    ax1.contour(walls[:,:,slice_index], extent = [0,35,0,35],alpha=0.1, colors='blue')
    ax1.contour(nodes[:,:,slice_index], extent = [0,35,0,35],alpha=0.2, colors='red')

    z_min = slice_index * dl
    z_max = (slice_index + 1) * dl
    mask = (coords_0[:,2] > z_min) & (coords_0[:,2] < z_max)

    x = coords_0[:, 0][mask]
    y = coords_0[:, 1][mask]
    z = coords_0[:, 2][mask]

    counts, xedges, yedges, im2 = ax2.hist2d(x, y, bins = 500, cmap = "Grays_r", norm = LogNorm(vmin = 1, vmax = 10**4))
    ax2.set_xlabel("cMpc/h")
    ax2.set_ylabel("cMpc/h")
    ax2.set_aspect('equal')
    ax2.contour(filaments[:,:,slice_index], extent = [0,35,0,35],alpha=0.1, colors='green', linewidths=3.0)
    ax2.contour(walls[:,:,slice_index], extent = [0,35,0,35],alpha=0.1, colors='blue')
    ax2.contour(nodes[:,:,slice_index], extent = [0,35,0,35],alpha=0.2, colors='red')

anim = animation.FuncAnimation(fig, update, frames=128, interval=50)

anim.save('psdtfe_vs_real.gif', writer='pillow', fps=10)

print("2D Animation successfully saved to GIF!")

In [ ]:
ids_0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['ParticleIDs'])

In [ ]:
bool_filament = filaments.astype(bool)
bool_wall = walls.astype(bool)
bool_node = nodes.astype(bool)
bool_void = voids.astype(bool)

In [ ]:
grid_x = np.floor(coords_0[:, 0] / dl).astype(int) % res
grid_y = np.floor(coords_0[:, 1] / dl).astype(int) % res
grid_z = np.floor(coords_0[:, 2] / dl).astype(int) % res

in_filament_mask = bool_filament[grid_x, grid_y, grid_z]


In [ ]:
filament_particle_ids = ids_0[in_filament_mask]
print(f"Total DM particles: {len(ids_0)}")
print(f"Particles in filaments: {len(filament_particle_ids)}")

In [ ]:
snap_z = 33 

particles_z = il.snapshot.loadSubset(basePath, snap_z, 'dm', ['Coordinates', 'ParticleIDs'])
coords_z = particles_z['Coordinates']/1000
ids_z = particles_z['ParticleIDs']

print("Matching particles across time (this might take a moment)...")
match_mask = np.isin(ids_z, filament_particle_ids)

historical_coords = coords_z[match_mask]
historical_ids = ids_z[match_mask] # Just to verify

print(f"Successfully recovered {len(historical_coords)} particles at z=.")


In [ ]:
slice_index = 1

#z_min = slice_index * dl
#z_max = (slice_index + 1) * dl
#mask = (historical_coords[:,2] > z_min) & (historical_coords[:,2] < z_max)

x = historical_coords[:, 0]#[mask]
y = historical_coords[:, 1]#[mask]
z = historical_coords[:, 2]#[mask]

plt.hist2d(x, y, bins = 500, cmap = "inferno", norm = LogNorm())
plt.xlabel('X (cMpc/h)')
plt.ylabel('Y (cMpc/h)')
plt.title('Forming Filaments')
plt.axis('square');

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)


def update(frame):

    ax.clear()

    snap_id = frame

    particles_z = il.snapshot.loadSubset(basePath, snap_id, 'dm', ['Coordinates', 'ParticleIDs'])
    coords_z = particles_z['Coordinates']/1000
    ids_z = particles_z['ParticleIDs']

    match_mask = np.isin(ids_z, filament_particle_ids)
    historical_coords = coords_z[match_mask]

    x = historical_coords[:, 0]
    y = historical_coords[:, 1]
    #z = historical_coords[:, 2]

    ax.hist2d(x, y, bins = 500, cmap = "inferno", norm = LogNorm())
    ax.set_xlabel('cMpc/h')
    ax.set_ylabel('cMpc/h')
    ax.set_title('Forming Filaments')
    ax.set_aspect('equal');

anim = animation.FuncAnimation(fig, update, frames=99, interval=50)

anim.save('formig_filaments.mp4', writer='ffmpeg', fps=10)

print("2D Animation successfully saved!")

In [ ]:
in_node_mask = bool_node[grid_x, grid_y, grid_z]

node_particle_ids = ids_0[in_node_mask]
print(f"Total DM particles: {len(ids_0)}")
print(f"Particles in nodes: {len(node_particle_ids)}")

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)


def update(frame):

    ax.clear()

    snap_id = frame

    particles_z = il.snapshot.loadSubset(basePath, snap_id, 'dm', ['Coordinates', 'ParticleIDs'])
    coords_z = particles_z['Coordinates']/1000
    ids_z = particles_z['ParticleIDs']

    match_mask = np.isin(ids_z, node_particle_ids)
    historical_coords = coords_z[match_mask]

    x = historical_coords[:, 0]
    y = historical_coords[:, 1]
    #z = historical_coords[:, 2]

    ax.hist2d(x, y, bins = 500, cmap = "inferno", norm = LogNorm())
    ax.set_xlabel('cMpc/h')
    ax.set_ylabel('cMpc/h')
    ax.set_title('Forming Nodes')
    ax.set_aspect('equal');

anim = animation.FuncAnimation(fig, update, frames=99, interval=50)

anim.save('formig_nodes.mp4', writer='ffmpeg', fps=10)

print("2D Animation successfully saved!")

In [ ]:
in_wall_mask = bool_wall[grid_x, grid_y, grid_z]

wall_particle_ids = ids_0[in_wall_mask]
print(f"Total DM particles: {len(ids_0)}")
print(f"Particles in walls: {len(wall_particle_ids)}")

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)


def update(frame):

    ax.clear()

    snap_id = frame

    particles_z = il.snapshot.loadSubset(basePath, snap_id, 'dm', ['Coordinates', 'ParticleIDs'])
    coords_z = particles_z['Coordinates']/1000
    ids_z = particles_z['ParticleIDs']

    match_mask = np.isin(ids_z, wall_particle_ids)
    historical_coords = coords_z[match_mask]

    x = historical_coords[:, 0]
    y = historical_coords[:, 1]
    #z = historical_coords[:, 2]

    ax.hist2d(x, y, bins = 500, cmap = "inferno", norm = LogNorm())
    ax.set_xlabel('cMpc/h')
    ax.set_ylabel('cMpc/h')
    ax.set_title('Forming Walls')
    ax.set_aspect('equal');

anim = animation.FuncAnimation(fig, update, frames=99, interval=50)

anim.save('formig_walls.mp4', writer='ffmpeg', fps=10)

print("2D Animation successfully saved to GIF!")

In [ ]:
in_void_mask = bool_void[grid_x, grid_y, grid_z]

void_particle_ids = ids_0[in_void_mask]
print(f"Total DM particles: {len(ids_0)}")
print(f"Particles in voids: {len(void_particle_ids)}")

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)


def update(frame):

    ax.clear()

    snap_id = frame

    particles_z = il.snapshot.loadSubset(basePath, snap_id, 'dm', ['Coordinates', 'ParticleIDs'])
    coords_z = particles_z['Coordinates']/1000
    ids_z = particles_z['ParticleIDs']

    match_mask = np.isin(ids_z, void_particle_ids)
    historical_coords = coords_z[match_mask]

    x = historical_coords[:, 0]
    y = historical_coords[:, 1]
    #z = historical_coords[:, 2]

    ax.hist2d(x, y, bins = 500, cmap = "inferno", norm = LogNorm())
    ax.set_xlabel('cMpc/h')
    ax.set_ylabel('cMpc/h')
    ax.set_title('Forming Voids')
    ax.set_aspect('equal');

anim = animation.FuncAnimation(fig, update, frames=99, interval=50)

anim.save('formig_voids.mp4', writer='ffmpeg', fps=10)

print("2D Animation successfully saved to GIF!")